In [306]:
import pandas as pd
from scipy.io import wavfile
from glob import glob
from tqdm import tqdm
import numpy as np
from scipy.signal import butter, filtfilt, welch, spectrogram

import os
import cv2
import h5py
import json
import pandas as pd
import shutil as sh
import seaborn as sns


from matplotlib.pyplot import *
from roboflow import Roboflow
import re
import copy
import supervision as sv

from inference import get_model
from scipy.signal import butter, sosfilt
import math



In [307]:
experiments = [387]
headmic_channels = [118,35]

In [ ]:
# Loading Ralph's model
# rf = Roboflow(api_key="CgU6vv2m60XHS8arrsvy")
# project = rf.workspace().project("gerbil-vox-bbox")
# model = project.version(6).model

model = get_model("gerbil-vox-bbox/6",api_key="CgU6vv2m60XHS8arrsvy")



label_annotator = sv.LabelAnnotator()
bounding_box_annotator = sv.BoxAnnotator()


: 

In [302]:
# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [atoi(c) for c in re.split(r'(\d+)', text)]



def write_spectrogram_img(audio, sr, path=None, lo=-70, hi=0, return_spectrogram=False):
    """
    Convert an audio signal into a spectrogram image and optionally return the spectrogram matrix.

    Args:
        audio (np.ndarray): 1D audio waveform samples.
        sr (int): Sampling rate of the audio signal in Hz.
        path (str, optional): Output file path for the spectrogram image. If None, image is not saved.
        lo (float, optional): Lower dB bound for clipping. Defaults to -70 dB.
        hi (float, optional): Upper dB bound for clipping. Defaults to 0 dB.
        return_spectrogram (bool, optional): If True, returns the Pxx_dB matrix (flipped and clipped). Defaults to False.

    Returns:
        freqs (np.ndarray): Frequency bins
        bins (np.ndarray): Time bins
        shape (tuple): Image shape (height, width)
        Pxx_dB (np.ndarray, optional): Flipped, clipped, and normalized spectrogram in dB (if return_spectrogram=True)
    """
    freqs, bins, Pxx = spectrogram(audio, fs=sr, nperseg=512, noverlap=256)

    # convert to dB and flip vertically for visualization
    Pxx_dB = np.flipud(10 * np.log10(Pxx + 1e-12))

    # clip and normalize
    a = np.clip(Pxx_dB, lo, hi)
    a = (a - lo) / (hi - lo + 1e-12)
    a = (a * 255).astype(np.uint8)

    if path is not None:
        cv2.imwrite(path, a, [cv2.IMWRITE_PNG_COMPRESSION, 0])

    if return_spectrogram:
        return freqs, bins, a.shape, Pxx_dB
    else:
        return freqs, bins, a.shape




def save_image_bboxes(img_path):
            
            img = cv2.imread(img_path)


            prediction = model.infer(img)[0]

            detections = sv.Detections.from_inference(prediction)

            filtered_detections = detections.with_nms(threshold=0.3, class_agnostic=True)


            annotated_image = bounding_box_annotator.annotate(scene=img, detections=filtered_detections)
                                                                
            cv2.imwrite(img_path.split(".")[0]+"_bbox.png", annotated_image)

            return filtered_detections


def bbox_to_time_freq(pred, freqs, bins, img_shape):

    h, w = img_shape

    # x = pred.x
    # y = pred.y
    # bw = pred.width
    # bh = pred.height

    # x1 = x - bw/2
    # x2 = x + bw/2

    # y1 = y - bh/2
    # y2 = y + bh/2

    x1 = pred[0]
    y1 = pred[1]
    x2 = pred[2]
    y2 = pred[3]

    # time mapping
    t1 = bins[min(int(x1 / w * len(bins)),len(bins)-1)]
    t2 = bins[min(int(x2 / w * len(bins)),len(bins)-1)]

    # frequency mapping (remember we flipped vertically)
    f1 = freqs[min(int((h - y2) / h * len(freqs)),len(freqs)-1)]
    f2 = freqs[min(int((h - y1) / h * len(freqs)),len(freqs)-1)]

    return t1, t2, f1, f2


def merge_events(events):
    """
    Merge overlapping or nearby events in time and keep only one event
    from each overlapping cluster.

    Preference order:
        1. Higher frequency (center frequency)
        2. Longer duration

    events: list of dicts with keys "t1", "t2", optionally "f1", "f2"
    gap: float, maximum allowed gap between events to consider them overlapping
    """

    

    # remove very short events
    events = [e for e in events if (e["t2"] - e["t1"]) >= 0.01]

    if not events:
        return []

    # Sort by start time
    events = sorted(events, key=lambda x: x["t1"])


    merged_clusters = []
    
    # last cluster
    merged_clusters.append(select_best_event(events))

    return merged_clusters


def select_best_event(cluster):
    """
    Select best event based on frequency preference, then duration.
    Events shorter than 0.01 s are discarded.
    """

    def score(e):
        duration = e["t2"] - e["t1"]

        if "f1" in e and "f2" in e:
            center_freq = (e["f1"] + e["f2"]) / 2
        else:
            center_freq = 0

        return (duration,center_freq)

    return max(cluster, key=score)



def bandpass_filter(audio, sr, f_low, f_high):
    """Bandpass filter audio to a frequency range."""
    sos = butter(4, [f_low, f_high], btype='band', fs=sr, output='sos')
    return sosfilt(sos, audio)


def save_event_cropped_spectrogram(headmic_audio, sr, start, event, out_path):

    new_start = start + event["t1"]
    new_stop = start + event["t2"]

    new_start_i = int(sr * new_start)
    new_stop_i = int(sr * new_stop)

    new_start_i = max(0, new_start_i)
    new_stop_i = min(len(headmic_audio), new_stop_i)

    cropped_audio = headmic_audio[new_start_i:new_stop_i]

    f1, f2 = event["f1"], event["f2"]

    if f1 >= f2 or f1 == 0 or f2 == 0:
        filtered_audio = cropped_audio
    else:
        filtered_audio = bandpass_filter(cropped_audio, sr, f1, f2)

    # signal power
    signal_power = np.mean(filtered_audio.astype(np.float64) ** 2)

    # noise estimate (rest of window)
    noise = np.concatenate([
        headmic_audio[max(0,new_start_i-2000):new_start_i],
        headmic_audio[new_stop_i:new_stop_i+2000]
    ])

    noise_power = np.mean(noise.astype(np.float64) ** 2) if len(noise) else 0

    snr = 10*np.log10(signal_power / noise_power) if noise_power > 0 else 0

    cropped_path = out_path.replace(".png", "_cropped.png")
    write_spectrogram_img(filtered_audio, sr, cropped_path)

    return signal_power, snr, f1, f2


### Saving spectrogram image files from DAS output times

In [303]:
no_vox = 0

for exp in experiments:
    general_path = f"D:/big_setup/experiment_{exp}/concatenated_data_cam_mic_sync/"
    img_path = os.path.join(general_path, "spectrograms")
    os.makedirs(img_path, exist_ok=True)

    # DAS files
    das_files = sorted(
        glob(f"//sanesstorage.cns.nyu.edu/archive/Niegil/das/training_data/data/exp_{exp}*annotations.csv"),
        key=natural_keys
    )

    # Headmic files
    headmic_files = {
        ch: sorted(glob(f"{general_path}headmic_{ch}_*.wav"), key=natural_keys)
        for ch in headmic_channels
    }

    length_of_experiment = len(glob(f"{general_path}*.mp4"))

    for idx in tqdm(range(length_of_experiment)):
        idx_str = "%03d"%(idx)

        # Load headmic audio
        headmics = {}
        for ch in headmic_channels:
            file = headmic_files[ch][idx]
            if idx_str not in file:
                raise Exception(f"Error finding headmic file ({file}) for idx {idx}")
            sr, headmics[ch] = wavfile.read(file)

        # Find DAS file
        matches = [f for f in das_files if idx_str in f]
        if len(matches) != 1:
            raise Exception(f"Expected 1 DAS file for index {idx}, found {len(matches)}")
        das_file = matches[0]

        vox_times = pd.read_csv(das_file)
        vox_rows = vox_times[vox_times["name"] == "vox"]
        base_name = os.path.basename(das_file).split(".")[0]


        vox_times["label"] = None
        vox_times["method"] = None

        for vox_idx, start, stop in zip(vox_rows.index, vox_rows.start_seconds, vox_rows.stop_seconds):
            no_vox += 1
            orig_start, orig_stop = start, stop

            # Sync adjustment
            add_amt = 0.05
            start -= add_amt / 2
            stop += add_amt / 2

            power_headmic = {}

            if math.isnan(start) or math.isnan(stop):
                continue

            # Process each headmic: generate full spectrogram and detect events
            for ch in headmic_channels:
                
                start_i = max(0, int(sr * start))
                stop_i = min(len(headmics[ch]), int(sr * stop))

                audio = headmics[ch][start_i:stop_i]
                out_path = os.path.join(img_path, f"{base_name}_img_{vox_idx}_mic_{ch}.png")

                # full spectrogram
                freqs, bins, shape, Sxx = write_spectrogram_img(audio, sr, out_path, return_spectrogram=True)

                # detect events
                pred = save_image_bboxes(out_path)
                events = []
                for p in pred.xyxy:
                    t1, t2, f1, f2 = bbox_to_time_freq(p, freqs, bins, shape)
                    events.append({"t1": t1, "t2": t2, "f1": f1, "f2": f2})
                events = merge_events(events)


                if events:
                    vals = save_event_cropped_spectrogram(headmics[ch],
                                                            sr,
                                                            start,
                                                            events[0],
                                                            out_path)
                    
                    if np.mean([vals[2],vals[3]]) < 10000: # discarding the detections below a certain frequency
                        continue
                    else:
                        power_headmic[ch] = vals
            
            if len(power_headmic) == 1 : # detection in one of the spectrograms

                best_ch, vals = next(iter(power_headmic.items()))

                if vals[0] > 0:
                    vox_times.loc[vox_idx, "label"] = best_ch
                    vox_times.loc[vox_idx, "method"] = "one_det"

            elif len(power_headmic) == 2: # detections are there in both spectrograms

                flag = 1
                for ch in headmic_channels:
                    if power_headmic[ch][1] == 0.0:
                        flag = 0

                best_ch, vals = max(power_headmic.items(), key=lambda x: x[1][flag])

                if vals[0] > 0:
                    vox_times.loc[vox_idx, "label"] = best_ch
                    if flag == 0:
                        vox_times.loc[vox_idx, "method"] = "snr"
                    else:
                        vox_times.loc[vox_idx, "method"] = "sig_pwr"

        vox_times.to_csv(das_file.split(".csv")[0]+"_gt.csv", index = False)


print(f"Total number of vocalizations: {no_vox}")

100%|██████████| 30/30 [03:44<00:00,  7.49s/it]


Total number of vocalizations: 664
